# Conformal coverage under English-to-Persian shift: scoring

This notebook runs a language model on paired English and Persian Global-MMLU questions.
For each question and language it saves the log-probabilities of options A, B, C and D.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

Results are saved to Google Drive after every chunk. If Colab disconnects, run all cells again and the
notebook continues from where it stopped.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
LIMIT = 2000      # number of question pairs; the order is fixed, so a larger LIMIT extends the same run
SAVE_EVERY = 10   # results are saved after this many pairs
OUTPUT_FOLDER = "conformal-language-shift"


In [ ]:
import json, os, random, sys, time
from pathlib import Path

import pyarrow.parquet as pq
import requests
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print("transformers", transformers.__version__, "| torch", torch.__version__)
ALLOW_CPU = globals().get("ALLOW_CPU", False)
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
elif ALLOW_CPU:
    DEVICE = "cpu"
else:
    raise RuntimeError("No GPU found. Choose Runtime > Change runtime type > T4 GPU, then run all cells again.")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive") / OUTPUT_FOLDER
else:
    OUTPUT_DIR = Path(globals().get("LOCAL_OUTPUT_DIR", OUTPUT_FOLDER))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / (MODEL_ID.rstrip("/").split("/")[-1] + ".jsonl")
print("Saving to", OUTPUT_FILE)


In [ ]:
DATA_DIR = Path(globals().get("LOCAL_DATA_DIR", "data"))
DATA_DIR.mkdir(exist_ok=True)
BASE = "https://huggingface.co/api/datasets/CohereLabs/Global-MMLU/parquet/{}/test/0.parquet"

def load_language(language):
    path = DATA_DIR / f"{language}_test.parquet"
    if not path.exists():
        response = requests.get(BASE.format(language), timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
    return pq.read_table(path).to_pylist()

english = {row["sample_id"]: row for row in load_language("en")}
pairs = [{"id": row["sample_id"], "en": english[row["sample_id"]], "fa": row} for row in load_language("fa")]
random.Random(0).shuffle(pairs)
assert len(pairs) == 14042, f"expected 14042 pairs, found {len(pairs)}"
print("pairs available:", len(pairs))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE).eval()

LETTERS = ["A", "B", "C", "D"]
LETTER_IDS = []
for letter in LETTERS:
    encoded = tokenizer.encode(letter, add_special_tokens=False)
    assert len(encoded) == 1, f"{letter} is not a single token"
    LETTER_IDS.append(encoded[0])
if DEVICE == "cuda":
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
INSTRUCTION = {
    "en": "Answer the following multiple-choice question. Reply with the letter of the correct option only.",
    "fa": "به پرسش چندگزینه‌ای زیر پاسخ بده. فقط حرف گزینهٔ درست را بنویس.",
}

def encode_prompt(row, language):
    options = "\n".join(f"{letter}. {row['option_' + letter.lower()]}" for letter in LETTERS)
    message = f"{INSTRUCTION[language]}\n\n{row['question']}\n{options}"
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": message}], add_generation_prompt=True, tokenize=False
    )
    return tokenizer(text, add_special_tokens=False)["input_ids"]

@torch.inference_mode()
def score_prompt(tokens):
    """Return (letter log-probabilities, letter mass) for one prompt.

    Prompts are scored one at a time, with no padding, so a score cannot depend on
    which other questions happen to share its batch.
    """
    ids = torch.tensor([tokens], device=DEVICE)
    output = model(input_ids=ids, logits_to_keep=1, use_cache=False)
    log_probs = torch.log_softmax(output.logits[0, -1].float(), dim=-1)[LETTER_IDS]
    assert torch.isfinite(log_probs).all(), "The model returned NaN or infinity."
    return log_probs.cpu().tolist(), log_probs.exp().sum().item()


In [ ]:
# Scoring the same prompt twice must give the same result.
sample = [encode_prompt(pair[language], language) for pair in pairs[:6] for language in ("en", "fa")]
first = [score_prompt(tokens) for tokens in sample]
second = [score_prompt(tokens) for tokens in sample]
largest = max(abs(a - b) for x, y in zip(first, second) for a, b in zip(x[0], y[0]))
print(f"largest difference between two runs: {largest:.6f}")
assert largest < 1e-3, "The same prompt gave different scores. Do not continue."
print("letter mass, English:", [round(first[i][1], 3) for i in range(0, 12, 2)])
print("letter mass, Persian:", [round(first[i][1], 3) for i in range(1, 12, 2)])
print("Self-check passed.")


In [ ]:
done = set()
if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open(encoding="utf-8") as source:
        done = {json.loads(line)["id"] for line in source if line.strip()}
todo = [pair for pair in pairs[:LIMIT] if pair["id"] not in done]
print(f"already saved: {len(done)} | to score: {len(todo)}")

started = time.time()
with OUTPUT_FILE.open("a", encoding="utf-8") as sink:
    for count, pair in enumerate(todo, start=1):
        record = {
            "id": pair["id"],
            "answer": LETTERS.index(pair["en"]["answer"]),
            "subject": pair["en"]["subject"],
            "category": pair["en"]["subject_category"],
            "cultural_label": pair["en"]["cultural_sensitivity_label"],
        }
        for language in ("en", "fa"):
            log_probs, mass = score_prompt(encode_prompt(pair[language], language))
            record[f"{language}_log_probs"] = log_probs
            record[f"{language}_letter_mass"] = mass
        sink.write(json.dumps(record) + "\n")
        if count % SAVE_EVERY == 0 or count == len(todo):
            sink.flush()
            os.fsync(sink.fileno())
            rate = count / (time.time() - started)
            minutes_left = (len(todo) - count) / rate / 60
            print(f"{count}/{len(todo)} pairs | {rate:.2f} pairs/s | about {minutes_left:.0f} min left", flush=True)
print("Scoring finished.")


In [ ]:
# A quick check only. The conformal analysis runs later on the laptop with src/analyze.py.
import statistics

with OUTPUT_FILE.open(encoding="utf-8") as source:
    rows = [json.loads(line) for line in source if line.strip()]
print("pairs saved:", len(rows), "| unique:", len({row["id"] for row in rows}))
for language in ("en", "fa"):
    correct = sum(
        max(range(4), key=lambda option: row[f"{language}_log_probs"][option]) == row["answer"] for row in rows
    )
    mass = statistics.median(row[f"{language}_letter_mass"] for row in rows)
    print(f"{language}: accuracy {correct / len(rows):.3f} | median letter mass {mass:.3f}")
print("\nDownload this file from Google Drive and put it in results/ on your laptop:")
print(OUTPUT_FILE)
